# Indikatoren-Analyse für ausgewählte Länder

Analyse der Differenzen (2015-2023) für:
- Portugal (PRT)
- Vanuatu (VUT)
- Botswana (BWA)
- Somalia (SOM)

In [1]:
import pandas as pd
import os
from pathlib import Path
import numpy as np

## 1. Einlesen aller Indikatoren-Dateien

In [2]:
# Pfad zum indicators-Verzeichnis
indicators_path = Path("data/vuln_indicators")

# Länder die analysiert werden sollen
countries = {
    'PRT': 'Portugal',
    'VUT': 'Vanuatu',
    'BWA': 'Botswana',
    'SOM': 'Somalia'
}

# Dictionary zum Speichern aller DataFrames
indicator_data = {}

# Erlaubte Präfixe für Vulnerability-Indikatoren
allowed_prefixes = ['id_food_', 'id_wate_', 'id_heal_', 'id_ecos_', 'id_habi_', 'id_infr_']

# Alle Ordner finden, die mit "id_" beginnen (vor Filter)
all_id_folders = [f for f in indicators_path.iterdir() if f.is_dir() and f.name.startswith('id_')]

# Nur Vulnerability-Indikatoren filtern
id_folders = sorted([f for f in all_id_folders if any(f.name.startswith(prefix) for prefix in allowed_prefixes)])

print(f"Alle id_-Ordner gefunden: {len(all_id_folders)}")
print(f"Vulnerability-Indikatoren (nach Filter): {len(id_folders)}")

# Alle score.csv Dateien einlesen
for folder in id_folders:
    input_file = folder / "score.csv"
    if input_file.exists():
        try:
            df = pd.read_csv(input_file)
            indicator_name = folder.name
            indicator_data[indicator_name] = df
            print(f"✓ {indicator_name}: {len(df)} Länder eingelesen")
        except Exception as e:
            print(f"✗ Fehler bei {folder.name}: {e}")

print(f"\nInsgesamt {len(indicator_data)} Vulnerability-Indikatoren erfolgreich eingelesen")

Alle id_-Ordner gefunden: 45
Vulnerability-Indikatoren (nach Filter): 36
✓ id_ecos_01: 192 Länder eingelesen
✓ id_ecos_02: 192 Länder eingelesen
✓ id_ecos_03: 192 Länder eingelesen
✓ id_ecos_04: 192 Länder eingelesen
✓ id_ecos_05: 192 Länder eingelesen
✓ id_ecos_06: 192 Länder eingelesen
✓ id_food_01: 192 Länder eingelesen
✓ id_food_02: 192 Länder eingelesen
✓ id_food_03: 192 Länder eingelesen
✓ id_food_04: 192 Länder eingelesen
✓ id_food_05: 192 Länder eingelesen
✓ id_food_06: 192 Länder eingelesen
✓ id_habi_01: 192 Länder eingelesen
✓ id_habi_02: 192 Länder eingelesen
✓ id_habi_03: 192 Länder eingelesen
✓ id_habi_04: 192 Länder eingelesen
✓ id_habi_05: 192 Länder eingelesen
✓ id_habi_06: 192 Länder eingelesen
✓ id_heal_01: 192 Länder eingelesen
✓ id_heal_02: 192 Länder eingelesen
✓ id_heal_03: 192 Länder eingelesen
✓ id_heal_04: 192 Länder eingelesen
✓ id_heal_05: 192 Länder eingelesen
✓ id_heal_06: 192 Länder eingelesen
✓ id_infr_01: 192 Länder eingelesen
✓ id_infr_02: 192 Länder ei

## 2. Berechnung der Differenzen (2015-2023)

In [3]:
# Dictionary für die Ergebnisse
results = {country_code: {} for country_code in countries.keys()}

# Für jeden Indikator die Differenzen berechnen
for indicator_name, df in indicator_data.items():
    # Prüfen ob die Jahre 2015 und 2023 als Spalten existieren
    if '2015' in df.columns and '2023' in df.columns:
        for country_code in countries.keys():
            # Land im DataFrame finden
            country_row = df[df['ISO3'] == country_code]
            
            if not country_row.empty:
                try:
                    value_2015 = pd.to_numeric(country_row['2015'].values[0], errors='coerce')
                    value_2023 = pd.to_numeric(country_row['2023'].values[0], errors='coerce')
                    
                    # Differenz berechnen (nur wenn beide Werte verfügbar sind)
                    if pd.notna(value_2015) and pd.notna(value_2023):
                        diff = value_2023 - value_2015
                        results[country_code][indicator_name] = {
                            '2015': value_2015,
                            '2023': value_2023,
                            'Differenz': diff
                        }
                except Exception as e:
                    pass  # Fehler überspringen

# Überblick über die Ergebnisse
for country_code in countries.keys():
    print(f"{countries[country_code]} ({country_code}): {len(results[country_code])} Indikatoren mit gültigen Daten")

Portugal (PRT): 36 Indikatoren mit gültigen Daten
Vanuatu (VUT): 27 Indikatoren mit gültigen Daten
Botswana (BWA): 33 Indikatoren mit gültigen Daten
Somalia (SOM): 30 Indikatoren mit gültigen Daten


## 3. Ranglisten erstellen

In [4]:
# Für jedes Land eine Rangliste nach absoluter Differenz erstellen
for country_code, country_name in countries.items():
    print(f"\n{'='*80}")
    print(f"RANGLISTE FÜR {country_name.upper()} ({country_code})")
    print(f"{'='*80}\n")
    
    # DataFrame aus den Ergebnissen erstellen
    country_results = results[country_code]
    
    if country_results:
        df_ranking = pd.DataFrame.from_dict(country_results, orient='index')
        
        # Nach absoluter Differenz sortieren (größte Veränderung zuerst)
        df_ranking['Abs_Differenz'] = df_ranking['Differenz'].abs()
        df_ranking = df_ranking.sort_values('Abs_Differenz', ascending=False)
        
        # Formatierung für bessere Lesbarkeit
        df_ranking['2015'] = df_ranking['2015'].round(4)
        df_ranking['2023'] = df_ranking['2023'].round(4)
        df_ranking['Differenz'] = df_ranking['Differenz'].round(4)
        df_ranking['Abs_Differenz'] = df_ranking['Abs_Differenz'].round(4)
        
        # Top 10 anzeigen
        print("Top 10 Indikatoren mit größter absoluter Veränderung:")
        print("-" * 80)
        display(df_ranking[['2015', '2023', 'Differenz', 'Abs_Differenz']].head(10))
        
        print(f"\n\nGesamtanzahl Indikatoren mit Daten: {len(df_ranking)}")
    else:
        print(f"Keine Daten verfügbar für {country_name}")


RANGLISTE FÜR PORTUGAL (PRT)

Top 10 Indikatoren mit größter absoluter Veränderung:
--------------------------------------------------------------------------------


,2015,2023,Differenz,Abs_Differenz
id_infr_06,0.6500,0.0750,-0.5750,0.5750
id_food_05,0.4711,0.6343,0.1632,0.1632
id_heal_06,0.1763,0.0733,-0.1030,0.1030
id_habi_05,0.4538,0.3590,-0.0948,0.0948
id_heal_05,0.6382,0.5648,-0.0733,0.0733
id_infr_03,0.8428,0.7772,-0.0656,0.0656
id_wate_03,0.1940,0.1381,-0.0560,0.0560
id_habi_04,0.5098,0.5638,0.0540,0.0540
id_food_04,0.3932,0.3459,-0.0473,0.0473
id_habi_03,0.6068,0.6541,0.0473,0.0473




Gesamtanzahl Indikatoren mit Daten: 36

RANGLISTE FÜR VANUATU (VUT)

Top 10 Indikatoren mit größter absoluter Veränderung:
--------------------------------------------------------------------------------


,2015,2023,Differenz,Abs_Differenz
id_infr_06,1.0000,0.4000,-0.6000,0.6000
id_infr_05,0.4820,0.3880,-0.0940,0.0940
id_habi_04,0.6970,0.7114,0.0144,0.0144
id_heal_05,0.9390,0.9521,0.0131,0.0131
id_food_04,0.8087,0.7978,-0.0109,0.0109
id_habi_03,0.1913,0.2022,0.0109,0.0109
id_ecos_05,0.5668,0.5684,0.0016,0.0016
id_ecos_04,0.1836,0.1831,-0.0005,0.0005
id_ecos_01,0.4588,0.4588,0.0000,0.0000
id_habi_06,0.7661,0.7661,0.0000,0.0000




Gesamtanzahl Indikatoren mit Daten: 27

RANGLISTE FÜR BOTSWANA (BWA)

Top 10 Indikatoren mit größter absoluter Veränderung:
--------------------------------------------------------------------------------


,2015,2023,Differenz,Abs_Differenz
id_infr_05,0.3829,0.2425,-0.1404,0.1404
id_habi_05,0.6169,0.4872,-0.1297,0.1297
id_food_03,0.1336,0.2540,0.1204,0.1204
id_infr_03,0.3493,0.2368,-0.1125,0.1125
id_infr_06,0.8750,0.9500,0.0750,0.0750
id_food_04,0.3540,0.2924,-0.0616,0.0616
id_habi_03,0.6460,0.7076,0.0616,0.0616
id_habi_04,0.5868,0.5447,-0.0421,0.0421
id_heal_04,0.4316,0.3996,-0.0320,0.0320
id_heal_03,0.2324,0.2095,-0.0229,0.0229




Gesamtanzahl Indikatoren mit Daten: 33

RANGLISTE FÜR SOMALIA (SOM)

Top 10 Indikatoren mit größter absoluter Veränderung:
--------------------------------------------------------------------------------


,2015,2023,Differenz,Abs_Differenz
id_habi_05,0.8883,0.7949,-0.0934,0.0934
id_heal_06,0.7371,0.6820,-0.0550,0.0550
id_food_04,0.6117,0.5613,-0.0504,0.0504
id_habi_03,0.3883,0.4387,0.0504,0.0504
id_habi_04,0.8934,0.8738,-0.0195,0.0195
id_ecos_04,0.1850,0.1942,0.0092,0.0092
id_infr_05,0.4931,0.5022,0.0091,0.0091
id_ecos_05,0.9754,0.9730,-0.0024,0.0024
id_food_05,0.9949,0.9946,-0.0003,0.0003
id_infr_02,0.0244,0.0244,0.0000,0.0000




Gesamtanzahl Indikatoren mit Daten: 30


## 4. Detaillierte Ranglisten nach positiven und negativen Veränderungen

In [5]:
# Zusätzliche Analyse: Positive vs. Negative Veränderungen
for country_code, country_name in countries.items():
    print(f"\n{'='*80}")
    print(f"DETAILANALYSE FÜR {country_name.upper()} ({country_code})")
    print(f"{'='*80}\n")
    
    country_results = results[country_code]
    
    if country_results:
        df_detail = pd.DataFrame.from_dict(country_results, orient='index')
        
        # Positive und negative Veränderungen trennen
        df_positive = df_detail[df_detail['Differenz'] > 0].sort_values('Differenz', ascending=False)
        df_negative = df_detail[df_detail['Differenz'] < 0].sort_values('Differenz', ascending=True)
        
        print(f"📈 TOP 5 GRÖSSTE POSITIVE VERÄNDERUNGEN (Anstieg):")
        print("-" * 80)
        if len(df_positive) > 0:
            display(df_positive[['2015', '2023', 'Differenz']].head(5))
        else:
            print("Keine positiven Veränderungen gefunden")
        
        print(f"\n📉 TOP 5 GRÖSSTE NEGATIVE VERÄNDERUNGEN (Rückgang):")
        print("-" * 80)
        if len(df_negative) > 0:
            display(df_negative[['2015', '2023', 'Differenz']].head(5))
        else:
            print("Keine negativen Veränderungen gefunden")
        
        print(f"\n📊 Zusammenfassung:")
        print(f"   • Positive Veränderungen: {len(df_positive)}")
        print(f"   • Negative Veränderungen: {len(df_negative)}")
        print(f"   • Keine Veränderung: {len(df_detail[df_detail['Differenz'] == 0])}")
    else:
        print(f"Keine Daten verfügbar für {country_name}")


DETAILANALYSE FÜR PORTUGAL (PRT)

📈 TOP 5 GRÖSSTE POSITIVE VERÄNDERUNGEN (Anstieg):
--------------------------------------------------------------------------------


,2015,2023,Differenz
id_food_05,0.471055,0.634277,0.163222
id_habi_04,0.509784,0.563800,0.054016
id_habi_03,0.606785,0.654118,0.047333
id_ecos_04,0.324324,0.333380,0.009056
id_food_03,0.235010,0.237729,0.002719



📉 TOP 5 GRÖSSTE NEGATIVE VERÄNDERUNGEN (Rückgang):
--------------------------------------------------------------------------------


,2015,2023,Differenz
id_infr_06,0.650000,0.075000,-0.575000
id_heal_06,0.176269,0.073259,-0.103010
id_habi_05,0.453813,0.358974,-0.094838
id_heal_05,0.638177,0.564844,-0.073333
id_infr_03,0.842773,0.777202,-0.065571



📊 Zusammenfassung:
   • Positive Veränderungen: 8
   • Negative Veränderungen: 8
   • Keine Veränderung: 20

DETAILANALYSE FÜR VANUATU (VUT)

📈 TOP 5 GRÖSSTE POSITIVE VERÄNDERUNGEN (Anstieg):
--------------------------------------------------------------------------------


...


📊 Zusammenfassung:
   • Positive Veränderungen: 6
   • Negative Veränderungen: 9
   • Keine Veränderung: 18

DETAILANALYSE FÜR SOMALIA (SOM)

📈 TOP 5 GRÖSSTE POSITIVE VERÄNDERUNGEN (Anstieg):
--------------------------------------------------------------------------------


,2015,2023,Differenz
id_habi_03,0.388343,0.438727,0.050383
id_ecos_04,0.184966,0.194198,0.009232
id_infr_05,0.493068,0.502161,0.009093



📉 TOP 5 GRÖSSTE NEGATIVE VERÄNDERUNGEN (Rückgang):
--------------------------------------------------------------------------------


,2015,2023,Differenz
id_habi_05,0.888278,0.794872,-0.093407
id_heal_06,0.737057,0.682008,-0.055049
id_food_04,0.611657,0.561273,-0.050383
id_habi_04,0.893365,0.873841,-0.019525
id_ecos_05,0.975387,0.972966,-0.002421



📊 Zusammenfassung:
   • Positive Veränderungen: 3
   • Negative Veränderungen: 6
   • Keine Veränderung: 21


In [ ]:
# Schweiz zur Länderliste hinzufügen
analysis_countries = {
    'CHE': 'Schweiz',
    'PRT': 'Portugal',
    'VUT': 'Vanuatu',
    'BWA': 'Botswana',
    'SOM': 'Somalia'
}

# Vulnerability-Daten für 2023 sammeln
vulnerability_2023 = {}

for country_code, country_name in analysis_countries.items():
    if country_code in results and results[country_code]:
        # Durchschnitt aller Vulnerability-Indikatoren für 2023
        values_2023 = [data['2023'] for data in results[country_code].values() if pd.notna(data['2023'])]
        if values_2023:
            vulnerability_2023[country_name] = np.mean(values_2023)
    else:
        # Daten direkt aus den Indikator-DataFrames holen
        values_2023 = []
        for df in indicator_data.values():
            if '2023' in df.columns:
                country_row = df[df['ISO3'] == country_code]
                if not country_row.empty:
                    value = pd.to_numeric(country_row['2023'].values[0], errors='coerce')
                    if pd.notna(value):
                        values_2023.append(value)
        if values_2023:
            vulnerability_2023[country_name] = np.mean(values_2023)

# GHG pro Kopf für 2023 einlesen
ghg_path = Path("data/vuln_indicators/id_ghgpc/score.csv")
ghg_2023 = {}

if ghg_path.exists():
    df_ghg = pd.read_csv(ghg_path)
    if '2023' in df_ghg.columns:
        for country_code, country_name in analysis_countries.items():
            country_row = df_ghg[df_ghg['ISO3'] == country_code]
            if not country_row.empty:
                value = pd.to_numeric(country_row['2023'].values[0], errors='coerce')
                if pd.notna(value):
                    ghg_2023[country_name] = value

# Ergebnisse anzeigen
print(f"\n{'='*80}")
print(f"VULNERABILITY UND GHG PRO KOPF FÜR 2023")
print(f"{'='*80}\n")

result_df = pd.DataFrame({
    'Land': list(analysis_countries.values()),
    'Durchschn. Vulnerability 2023': [vulnerability_2023.get(name, np.nan) for name in analysis_countries.values()],
    'GHG pro Kopf 2023': [ghg_2023.get(name, np.nan) for name in analysis_countries.values()]
})

result_df = result_df.round(4)
display(result_df)


VULNERABILITY UND GHG PRO KOPF FÜR 2023



,Land,Durchschn. Vulnerability 2023,GHG pro Kopf 2023
0,Schweiz,0.2503,NaN
1,Portugal,0.3185,NaN
2,Vanuatu,0.5313,NaN
3,Botswana,0.4344,NaN
4,Somalia,0.6244,NaN
